In [1]:
# ================= NOTEBOOK 1: DATA PRELOADING & PREPROCESSING =================
import os
import json
import urllib.request
import pandas as pd
import numpy as np

# 1. Download HateXplain raw files if not already present
os.makedirs("/content/hatexplain_data", exist_ok=True)
base_url = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Dataset/"

if not os.path.exists("/content/hatexplain_data/dataset.json"):
    urllib.request.urlretrieve(base_url + "dataset.json", "/content/hatexplain_data/dataset.json")
if not os.path.exists("/content/hatexplain_data/post_id_divisions.json"):
    urllib.request.urlretrieve(base_url + "post_id_divisions.json", "/content/hatexplain_data/post_id_divisions.json")

# 2. Load JSON data
with open("/content/hatexplain_data/dataset.json", "r") as f:
    dataset_raw = json.load(f)
with open("/content/hatexplain_data/post_id_divisions.json", "r") as f:
    divisions = json.load(f)

train_ids = set(divisions.get("train", []))
val_ids = set(divisions.get("val", []))
test_ids = set(divisions.get("test", []))

rows = []
for post_id, data in dataset_raw.items():
    if not isinstance(data, dict):
        continue
    # Extract text/tokens
    post_tokens = data.get("post_idioms", data.get("post_tokens", []))
    if len(post_tokens) == 0:
        continue
    text = " ".join(post_tokens)
    
    # Majority label logic from annotators
    annotators = data.get("annotators", [])
    labels = [ann.get("label") for ann in annotators if isinstance(ann, dict)]
    if len(labels) == 0:
        continue
    from collections import Counter
    maj_label = Counter(labels).most_common(1)[0][0]
    
    # Determine split
    if post_id in train_ids:
        split = "train"
    elif post_id in val_ids:
        split = "val"
    elif post_id in test_ids:
        split = "test"
    else:
        split = "train"
        
    rows.append({
        "post_id": post_id,
        "text": text,
        "post_tokens": post_tokens,
        "label": maj_label,
        "annotators": annotators,
        "rationales": data.get("rationales", []),
        "split": split
    })

df = pd.DataFrame(rows)
train = df[df["split"] == "train"].reset_index(drop=True)
val = df[df["split"] == "val"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)

# Create binary target y (normal = 0, hatespeech + offensive = 1)
for d in [train, val, test]:
    d["y"] = (d["label"] != "normal").astype(int)

print(f"Dataset Loaded Successfully!")
print(f"Train shape: {train.shape} | Val shape: {val.shape} | Test shape: {test.shape}")
print(f"Label distribution in train:\n{train['label'].value_counts()}")

ModuleNotFoundError: No module named 'pandas'